# Targeted video super-resolution with a frozen video-diffusion prior
IE 643 course project · Team Pairadox · self-contained Kaggle notebook

**Settings:** Accelerator **GPU T4 x2**, Internet **On**, Add-ons → Secrets → `HF_TOKEN`
(attached to this notebook). Run top to bottom once up to the timing section, then
**Save Version → Save & Run All**.

Part A writes the project's code into `/kaggle/working/tsr` (one cell per file).
Part B runs the experiments.

### External resources used (cite these in the report)
| What | Source | Role |
|---|---|---|
| DAVIS 2017 TrainVal 480p | davischallenge.org (Pont-Tuset et al., 2017) | evaluation data + object masks |
| Stable Video Diffusion img2vid | Hugging Face `stabilityai/stable-video-diffusion-img2vid` (Blattmann et al., 2023) | frozen video prior (our method) |
| Stable Diffusion ×4 upscaler | Hugging Face `stabilityai/stable-diffusion-x4-upscaler` | image-prior baseline (Exp 1) |
| Real-ESRGAN x4plus weights | github.com/xinntao/Real-ESRGAN (Wang et al., 2021) | GAN baseline |
| diffusers, spandrel, lpips, pyiqa, OpenCV, scikit-image | their GitHub/PyPI pages | libraries |
| Low-frequency colour fix idea | StableSR (Wang et al., 2024) | step 3 of our method |
| SDEdit partial-noising idea | Meng et al., ICLR 2022 | step 1 of our method |


In [ ]:
# ---- parameters: edit these ----
EXP1_VIDEO = ""         # path of your Exp 1 mp4 under /kaggle/input (use the copy-path icon), or "" to skip
N_TEST, N_VAL = 8, 3    # DAVIS objects for test / for tuning the strength
STRENGTHS = [0.3, 0.5, 0.7]
CLIP_LEN = 27           # two SVD windows (14 + 13 new frames)


In [ ]:
import os, shutil, subprocess, json, glob
import numpy as np
os.makedirs('/kaggle/working/tsr', exist_ok=True)
os.chdir('/kaggle/working/tsr')
for d in ['tsr/baselines', 'scripts', 'configs', 'tests']:
    os.makedirs(d, exist_ok=True)
print(os.getcwd())


## Part A · project code (run each cell once)


In [ ]:
%%writefile tsr/__init__.py
# package marker


In [ ]:
%%writefile tsr/io.py
"""Frame I/O. All frames are RGB uint8 numpy arrays of shape (H, W, 3)."""
from pathlib import Path

import cv2
import numpy as np

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv", ".webm"}


def read_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"Could not read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def write_image(path, img):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if img.ndim == 2:
        cv2.imwrite(str(path), img)
    else:
        cv2.imwrite(str(path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


def image_files(folder):
    return sorted(f for f in Path(folder).iterdir() if f.suffix.lower() in IMG_EXT)


def read_frames(path, max_frames=None, start=0):
    """Read a clip from a folder of images or a video file.

    Returns (frames, names) where names are frame stems used for output files.
    Frames are always consecutive (no subsampling) so temporal metrics are valid.
    """
    path = Path(path)
    if path.is_dir():
        files = image_files(path)[start:]
        if max_frames:
            files = files[:max_frames]
        return [read_image(f) for f in files], [f.stem for f in files]

    if path.suffix.lower() not in VID_EXT:
        raise ValueError(f"Unsupported input: {path}")
    cap = cv2.VideoCapture(str(path))
    frames, idx = [], 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if idx >= start:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if max_frames and len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames, [f"{i:05d}" for i in range(start, start + len(frames))]


def read_dir(folder, grayscale=False):
    files = image_files(folder)
    if grayscale:
        return [cv2.imread(str(f), cv2.IMREAD_GRAYSCALE) for f in files]
    return [read_image(f) for f in files]


def write_frames(folder, frames, names=None):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for i, frame in enumerate(frames):
        name = names[i] if names else f"{i:05d}"
        write_image(folder / f"{name}.png", frame)


def to_uint8(x):
    return np.clip(np.rint(x), 0, 255).astype(np.uint8)


In [ ]:
%%writefile tsr/degrade.py
"""Synthetic degradations that turn an HR tube into its LR input.

Two presets:
  bicubic   - antialiased ×4 bicubic only (the classic "BI" setting; fair to
              BasicVSR++, which was trained on it).
  realworld - Real-ESRGAN-style two-stage blur / resize / noise / JPEG with the
              op order shuffled per clip, plus optional H.264 compression.

Key design choice for video: every parameter (kernel, scale factors, noise
level, JPEG quality, CRF) is sampled ONCE per clip and reused for every frame.
Only the noise realisation changes per frame. Sampling per frame would create
artificial flicker in the LR input and unfairly punish temporal models.
"""
import os
import subprocess
import tempfile
import warnings

import cv2
import numpy as np
from PIL import Image

from .io import to_uint8

INTERPS = {"area": cv2.INTER_AREA, "linear": cv2.INTER_LINEAR, "cubic": cv2.INTER_CUBIC}

REALWORLD = dict(
    shuffle=True,
    blur_prob=1.0, blur_sigma1=(0.2, 2.5), blur_sigma2=(0.2, 1.2),
    aniso_prob=0.5, kernel_size=(7, 15),
    resize_range1=(0.5, 1.2),   # stage 1: relative to the HR size
    resize_range2=(0.75, 1.25), # stage 2: relative to the final LR size
    noise_prob=0.9, gauss_sigma=(1.0, 15.0), poisson_prob=0.4,
    poisson_scale=(0.05, 2.0), gray_noise_prob=0.4,
    jpeg_prob=0.9, jpeg_q=(40, 95),
    second_stage_prob=0.7,
    video_prob=0.5, crf=(18, 30),
)
# Lighter than the Real-ESRGAN paper on purpose: our LR targets are only
# 32-128 px, and the paper's settings can erase them completely.

PRESETS = {"realworld": REALWORLD, "bicubic": None}


# ----------------------------------------------------------------- sampling
def _sample_stage(rng, cfg, stage):
    p = {"order": ["blur", "resize", "noise", "jpeg"]}
    if rng.random() < cfg["blur_prob"]:
        lo, hi = cfg["kernel_size"]
        size = int(rng.integers(lo // 2, hi // 2 + 1)) * 2 + 1
        sig = cfg[f"blur_sigma{stage}"]
        sx = float(rng.uniform(*sig))
        sy = float(rng.uniform(*sig)) if rng.random() < cfg["aniso_prob"] else sx
        p["blur"] = [size, sx, sy, float(rng.uniform(0, np.pi))]
    p["resize"] = [float(rng.uniform(*cfg[f"resize_range{stage}"])),
                   str(rng.choice(list(INTERPS)))]
    if rng.random() < cfg["noise_prob"]:
        gray = bool(rng.random() < cfg["gray_noise_prob"])
        if rng.random() < cfg["poisson_prob"]:
            p["noise"] = ["poisson", float(rng.uniform(*cfg["poisson_scale"])), gray]
        else:
            p["noise"] = ["gauss", float(rng.uniform(*cfg["gauss_sigma"])), gray]
    if rng.random() < cfg["jpeg_prob"]:
        p["jpeg"] = int(rng.integers(cfg["jpeg_q"][0], cfg["jpeg_q"][1] + 1))
    if cfg["shuffle"]:
        rng.shuffle(p["order"])
    return p


def sample_params(rng, preset="realworld"):
    """Sample one clip's degradation. Returns a JSON-serialisable dict."""
    cfg = PRESETS[preset]
    if cfg is None:
        return {"preset": "bicubic", "stages": [], "final_interp": "pil_bicubic", "crf": None}
    stages = [_sample_stage(rng, cfg, 1)]
    if rng.random() < cfg["second_stage_prob"]:
        stages.append(_sample_stage(rng, cfg, 2))
    crf = int(rng.integers(cfg["crf"][0], cfg["crf"][1] + 1)) if rng.random() < cfg["video_prob"] else None
    return {"preset": preset, "stages": stages,
            "final_interp": str(rng.choice(list(INTERPS))), "crf": crf}


# ----------------------------------------------------------------- ops
def gaussian_kernel(size, sx, sy, theta):
    ax = np.arange(size) - size // 2
    xx, yy = np.meshgrid(ax, ax)
    c, s = np.cos(theta), np.sin(theta)
    xr, yr = c * xx + s * yy, -s * xx + c * yy
    k = np.exp(-0.5 * ((xr / sx) ** 2 + (yr / sy) ** 2))
    return (k / k.sum()).astype(np.float32)


def _add_noise(img, spec, rng):
    kind, level, gray = spec
    h, w = img.shape[:2]
    if kind == "gauss":
        shape = (h, w, 1) if gray else (h, w, 3)
        return img + rng.normal(0.0, level, shape).astype(np.float32)
    # Poisson (shot) noise: lower peak = stronger noise.
    peak = 255.0 / level
    src = img.mean(axis=2, keepdims=True) if gray else img
    src01 = np.clip(src / 255.0, 0, 1)
    noise = (rng.poisson(src01 * peak) / peak - src01) * 255.0
    return img + noise.astype(np.float32)


def _jpeg(img, quality):
    bgr = cv2.cvtColor(to_uint8(img), cv2.COLOR_RGB2BGR)
    ok, buf = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, int(quality)])
    return cv2.cvtColor(cv2.imdecode(buf, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB).astype(np.float32)


def _resize(img, wh, interp):
    wh = (max(8, int(round(wh[0]))), max(8, int(round(wh[1]))))
    return cv2.resize(img, wh, interpolation=INTERPS[interp])


def _apply_stage(img, p, stage_idx, lr_wh, rng):
    for op in p["order"]:
        if op == "blur" and "blur" in p:
            img = cv2.filter2D(img, -1, gaussian_kernel(*p["blur"]), borderType=cv2.BORDER_REFLECT)
        elif op == "resize":
            f, interp = p["resize"]
            h, w = img.shape[:2]
            target = (w * f, h * f) if stage_idx == 0 else (lr_wh[0] * f, lr_wh[1] * f)
            img = _resize(img, target, interp)
        elif op == "noise" and "noise" in p:
            img = _add_noise(img, p["noise"], rng)
        elif op == "jpeg" and "jpeg" in p:
            img = _jpeg(img, p["jpeg"])
    return np.clip(img, 0, 255)


def h264_roundtrip(frames, crf, preset="medium"):
    """Encode/decode frames with libx264 so the LR input has real codec artefacts."""
    h, w = frames[0].shape[:2]
    if h % 2 or w % 2:
        raise ValueError("H.264 (yuv420p) needs even frame sizes")
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "clip.mp4")
        enc = ["ffmpeg", "-loglevel", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
               "-s", f"{w}x{h}", "-r", "25", "-i", "-", "-c:v", "libx264", "-preset", preset,
               "-crf", str(crf), "-pix_fmt", "yuv420p", path]
        subprocess.run(enc, input=b"".join(f.tobytes() for f in frames), check=True)
        dec = ["ffmpeg", "-loglevel", "error", "-i", path, "-f", "rawvideo", "-pix_fmt", "rgb24", "-"]
        raw = subprocess.run(dec, capture_output=True, check=True).stdout
    out = np.frombuffer(raw, np.uint8).reshape(-1, h, w, 3)
    if len(out) != len(frames):
        raise RuntimeError(f"ffmpeg returned {len(out)} frames, expected {len(frames)}")
    return list(out.copy())


def degrade_clip(frames, params, scale, rng):
    """Degrade a list of HR uint8 frames into LR uint8 frames of size (H/scale, W/scale)."""
    h, w = frames[0].shape[:2]
    if h % scale or w % scale:
        raise ValueError(f"HR size {w}x{h} is not divisible by scale {scale}")
    lr_wh = (w // scale, h // scale)
    out = []
    for frame in frames:
        if params["final_interp"] == "pil_bicubic":
            # PIL bicubic is antialiased, close to MATLAB imresize (the BI protocol).
            out.append(np.asarray(Image.fromarray(frame).resize(lr_wh, Image.BICUBIC)))
            continue
        img = frame.astype(np.float32)
        for i, stage in enumerate(params["stages"]):
            img = _apply_stage(img, stage, i, lr_wh, rng)
        img = cv2.resize(img, lr_wh, interpolation=INTERPS[params["final_interp"]])
        out.append(to_uint8(img))

    if params.get("crf") is not None:
        try:
            out = h264_roundtrip(out, params["crf"])
        except (FileNotFoundError, subprocess.CalledProcessError) as e:
            warnings.warn(f"ffmpeg unavailable or failed ({e}); skipping video compression")
            params["crf"] = None  # record what was actually applied
    return out


In [ ]:
%%writefile tsr/tubes.py
"""Turn per-frame target masks into fixed-size, temporally smooth crop "tubes".

Rules that matter for fair evaluation:
  * The crop SIDE is constant over a window; only the centre moves. A per-frame
    size change would zoom in and out, which looks like flicker.
  * The HR ground truth is never upsampled. If the target is bigger than
    max_side we downscale the HR crop (keeps it sharp); if it is smaller than
    min_side we drop it (its LR version would be too small to be meaningful).
  * Side and padding are multiples of the SR scale (and of `align`), so the LR
    tube is an exact 1/scale of the HR tube and fits a latent-diffusion VAE
    (/8) at LR resolution.
"""
import math

import cv2
import numpy as np


def mask_to_box(mask):
    if mask is None:
        return None
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


def fill_gaps(boxes, max_gap):
    """Linearly interpolate missing boxes inside gaps of at most max_gap frames."""
    boxes = list(boxes)
    valid = [i for i, b in enumerate(boxes) if b is not None]
    for a, b in zip(valid, valid[1:]):
        gap = b - a - 1
        if 0 < gap <= max_gap:
            for k in range(1, gap + 1):
                t = k / (gap + 1)
                boxes[a + k] = [(1 - t) * u + t * v for u, v in zip(boxes[a], boxes[b])]
    return boxes


def contiguous_runs(boxes):
    runs, start = [], None
    for i, b in enumerate(boxes + [None]):
        if b is not None and start is None:
            start = i
        elif b is None and start is not None:
            runs.append((start, i))
            start = None
    return runs


def windows(start, end, length, min_len, max_windows=None):
    out = []
    for s in range(start, end, length):
        e = min(s + length, end)
        if e - s >= min_len:
            out.append((s, e))
    return out[:max_windows] if max_windows else out


def smooth(x, win):
    x = np.asarray(x, dtype=np.float64)
    if win <= 1 or len(x) < 3:
        return x
    win = min(win, len(x) if len(x) % 2 else len(x) - 1)
    pad = win // 2
    xp = np.pad(x, pad, mode="edge")
    return np.convolve(xp, np.ones(win) / win, mode="valid")


def _ceil_to(v, m):
    return int(math.ceil(v / m) * m)


def _floor_to(v, m):
    return int(math.floor(v / m) * m)


def plan_window(boxes, frame_hw, *, scale=4, margin=1.25, align=32,
                min_side=128, max_side=512, pad=16, smooth_win=5):
    """Plan a square crop for every frame of a window.

    Returns a dict (JSON-serialisable) or None if the target is too small.
      side      HR tube side after any downscaling (multiple of align)
      side_raw  side of the crop taken from the original frame
      pad/pad_raw  context padding used only during degradation
      x0, y0    per-frame top-left of the crop in original-frame pixels
    """
    assert align % scale == 0 and pad % scale == 0
    H, W = frame_hw
    b = np.asarray(boxes, dtype=np.float64)
    cx = smooth((b[:, 0] + b[:, 2]) / 2, smooth_win)
    cy = smooth((b[:, 1] + b[:, 3]) / 2, smooth_win)
    extent = np.maximum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1])
    side_raw = int(math.ceil(margin * smooth(extent, smooth_win).max()))
    side_raw = min(side_raw, min(H, W))
    if side_raw < min_side:
        return None

    if side_raw <= max_side:
        up = _ceil_to(side_raw, align)
        side = up if up <= min(H, W) else _floor_to(side_raw, align)
        side_raw, factor = side, 1.0
    else:
        side, factor = max_side, max_side / side_raw
    pad_raw = int(round(pad / factor))

    x0 = np.clip(np.rint(cx - side_raw / 2), 0, W - side_raw).astype(int)
    y0 = np.clip(np.rint(cy - side_raw / 2), 0, H - side_raw).astype(int)
    return dict(side=side, side_raw=side_raw, factor=factor, pad=pad, pad_raw=pad_raw,
                x0=x0.tolist(), y0=y0.tolist(), frame_hw=[H, W], scale=scale)


def crop_reflect(img, x0, y0, w, h):
    """Crop with reflect padding wherever the window leaves the frame."""
    H, W = img.shape[:2]
    l, t = max(0, -x0), max(0, -y0)
    r, btm = max(0, x0 + w - W), max(0, y0 + h - H)
    if l or t or r or btm:
        widths = ((t, btm), (l, r)) + ((0, 0),) * (img.ndim - 2)
        img = np.pad(img, widths, mode="reflect")
        x0, y0 = x0 + l, y0 + t
    return img[y0:y0 + h, x0:x0 + w]


def extract_padded(frame, plan, i):
    """HR window with context padding, at tube resolution: (side+2pad)^2."""
    sr, pr = plan["side_raw"], plan["pad_raw"]
    win = crop_reflect(frame, plan["x0"][i] - pr, plan["y0"][i] - pr, sr + 2 * pr, sr + 2 * pr)
    out = plan["side"] + 2 * plan["pad"]
    if win.shape[0] != out:
        win = cv2.resize(win, (out, out), interpolation=cv2.INTER_AREA)
    return win


def extract_mask(mask, plan, i):
    sr = plan["side_raw"]
    m = mask[plan["y0"][i]:plan["y0"][i] + sr, plan["x0"][i]:plan["x0"][i] + sr].astype(np.uint8)
    if sr != plan["side"]:
        m = cv2.resize(m, (plan["side"], plan["side"]), interpolation=cv2.INTER_NEAREST)
    return m


def unpad(img, pad):
    return img[pad:img.shape[0] - pad, pad:img.shape[1] - pad] if pad else img


In [ ]:
%%writefile tsr/targets.py
"""Where per-frame target masks come from.

  masks   - annotation PNGs (DAVIS layout: palette PNG, pixel value = object id)
  sam2    - SAM 2 video predictor, prompted with boxes on frame 0
            (boxes from a JSON file, or from a YOLO detector)
  box_track - OpenCV MIL tracker from a frame-0 box, as in Experiment 1.
            Masks are the tracked rectangles. Kept so the final results can be
            compared like-for-like with Exp 1, and as the SAM 2 fallback.
  whole   - the whole (centre-square) frame is the target, e.g. CelebV-HQ,
            whose clips are already face-centred

Every function returns {obj_id: [bool mask (H, W) or None per frame]}.
"""
import json
import tempfile
from pathlib import Path

import cv2
import numpy as np
from PIL import Image


# ------------------------------------------------------------- annotations
def masks_from_annotations(mask_dir, names, min_pixels=64):
    mask_dir = Path(mask_dir)
    per_frame = []
    for n in names:
        f = mask_dir / f"{n}.png"
        per_frame.append(np.array(Image.open(f)) if f.exists() else None)
    ids = set()
    for m in per_frame:
        if m is not None:
            if m.ndim == 3:  # RGB-coded annotations: collapse colours to ids
                raise ValueError(f"{mask_dir}: expected palette/index PNGs, got RGB")
            ids.update(int(v) for v in np.unique(m) if v not in (0, 255))
    out = {}
    for oid in sorted(ids):
        seq = []
        for m in per_frame:
            mm = None if m is None else (m == oid)
            seq.append(mm if mm is not None and mm.sum() >= min_pixels else None)
        out[oid] = seq
    return out


# ------------------------------------------------------------- whole frame
def whole_frame_target(frames):
    h, w = frames[0].shape[:2]
    m = np.ones((h, w), dtype=bool)
    return {1: [m] * len(frames)}


# ------------------------------------------------------------- boxes
def boxes_from_json(path, clip_id):
    """JSON: {"clip_id": [[x, y, w, h], ...]}  (boxes on frame 0, like the demo)."""
    data = {safe_id(k): v for k, v in json.loads(Path(path).read_text()).items()}
    clip_id = safe_id(clip_id)
    # exact match, else prefix: "6655841" matches "6655841-hd_1920_1080_30fps_3"
    key = clip_id if clip_id in data else next((k for k in data if clip_id.startswith(k)), None)
    if key is None:
        return {}
    return {i + 1: [x, y, x + w, y + h] for i, (x, y, w, h) in enumerate(data[key])}


def safe_id(name):
    """Clip ids from file names like '6655841-hd_1920_1080_30fps (3).mp4'."""
    return "".join(c if c.isalnum() or c in "-_" else "_" for c in name).strip("_")


def track_boxes_mil(frames, boxes0):
    """{obj: [x0, y0, x1, y1]} on frame 0 -> {obj: [rect mask per frame]}."""
    H, W = frames[0].shape[:2]
    out = {}
    for oid, (x0, y0, x1, y1) in boxes0.items():
        tracker = cv2.TrackerMIL_create()
        tracker.init(cv2.cvtColor(frames[0], cv2.COLOR_RGB2BGR),
                     (int(x0), int(y0), int(x1 - x0), int(y1 - y0)))
        seq, last = [], (int(x0), int(y0), int(x1 - x0), int(y1 - y0))
        for t, f in enumerate(frames):
            if t > 0:
                ok, box = tracker.update(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
                last = tuple(int(v) for v in box) if ok else last
            x, y, w, h = last
            m = np.zeros((H, W), bool)
            m[max(0, y):min(H, y + h), max(0, x):min(W, x + w)] = True
            seq.append(m if m.any() else None)
        out[oid] = seq
    return out


_YOLO = None


def detect_boxes(frame, conf=0.5, min_area=0.003, max_area=0.25, max_objects=2,
                 model_name="yolov8n.pt"):
    """Pick a few medium-sized, confidently detected objects on frame 0."""
    global _YOLO
    from ultralytics import YOLO
    if _YOLO is None:
        _YOLO = YOLO(model_name)
    res = _YOLO(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR), verbose=False)[0]
    H, W = frame.shape[:2]
    cands = []
    for (x0, y0, x1, y1), c in zip(res.boxes.xyxy.cpu().numpy(), res.boxes.conf.cpu().numpy()):
        a = (x1 - x0) * (y1 - y0) / (H * W)
        if c >= conf and min_area <= a <= max_area:
            cands.append((float(c), [float(x0), float(y0), float(x1), float(y1)]))
    cands.sort(key=lambda t: -t[0])
    return {i + 1: box for i, (_, box) in enumerate(cands[:max_objects])}


# ------------------------------------------------------------- SAM 2
class SAM2Tracker:
    """Thin wrapper over the official SAM 2 video predictor.

    pip install "git+https://github.com/facebookresearch/sam2.git"
    """

    def __init__(self, model_id="facebook/sam2.1-hiera-small", device="cuda"):
        import torch
        from sam2.sam2_video_predictor import SAM2VideoPredictor
        self.torch = torch
        self.device = device
        self.pred = SAM2VideoPredictor.from_pretrained(model_id, device=device)

    def track(self, frames, boxes0):
        if not boxes0:
            return {}
        torch = self.torch
        T = len(frames)
        out = {oid: [None] * T for oid in boxes0}
        use_bf16 = self.device.startswith("cuda") and torch.cuda.is_bf16_supported()
        with tempfile.TemporaryDirectory() as tmp:
            for i, f in enumerate(frames):  # SAM 2 expects a folder of numbered JPEGs
                cv2.imwrite(f"{tmp}/{i:05d}.jpg", cv2.cvtColor(f, cv2.COLOR_RGB2BGR),
                            [cv2.IMWRITE_JPEG_QUALITY, 95])
            # bf16 autocast on Ampere+; plain fp32 on a T4 (no fast bf16 there).
            ctx = torch.autocast("cuda", dtype=torch.bfloat16) if use_bf16 else _null()
            with torch.inference_mode(), ctx:
                state = self.pred.init_state(video_path=tmp, offload_video_to_cpu=True,
                                             offload_state_to_cpu=True)
                for oid, box in boxes0.items():
                    self.pred.add_new_points_or_box(inference_state=state, frame_idx=0,
                                                    obj_id=oid, box=np.asarray(box, np.float32))
                for fidx, obj_ids, logits in self.pred.propagate_in_video(state):
                    for k, oid in enumerate(obj_ids):
                        m = (logits[k, 0] > 0).float().cpu().numpy().astype(bool)
                        out[oid][fidx] = m if m.any() else None
                self.pred.reset_state(state)
        return out


class _null:
    def __enter__(self):
        return self

    def __exit__(self, *a):
        return False


In [ ]:
%%writefile tsr/metrics.py
"""Metrics on the target tube, each reported on the whole crop and inside the mask.

  PSNR / SSIM  Y channel (BT.601), as in the VSR literature (BasicVSR++, REDS4)
  LPIPS        AlexNet, spatial map averaged over the region
  tLPIPS       |LPIPS(sr_t, sr_t-1) - LPIPS(gt_t, gt_t-1)|  (TecoGAN definition)
  warp error   ||sr_t - warp(sr_t-1)||^2 using flow estimated on the GROUND TRUTH,
               over non-occluded mask pixels. Using GT flow means every method is
               judged with the same motion, and flow quality no longer depends on
               the method's own artefacts (the problem in Experiment 1).
"""
import cv2
import numpy as np
from skimage.metrics import structural_similarity


def rgb_to_y(img):
    img = img.astype(np.float64)
    return 16.0 + (65.481 * img[..., 0] + 128.553 * img[..., 1] + 24.966 * img[..., 2]) / 255.0


def psnr(a, b, region=None):
    d = (a.astype(np.float64) - b.astype(np.float64)) ** 2
    mse = d[region].mean() if region is not None else d.mean()
    return 100.0 if mse == 0 else float(10 * np.log10(255.0 ** 2 / mse))


def ssim(a_y, b_y, region=None):
    val, smap = structural_similarity(a_y, b_y, data_range=255, full=True,
                                      gaussian_weights=True, sigma=1.5,
                                      use_sample_covariance=False)
    return float(smap[region].mean()) if region is not None else float(val)


class LPIPS:
    """Lazy LPIPS so the rest works without torch."""

    def __init__(self, net="alex", device=None):
        import lpips
        import torch
        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = lpips.LPIPS(net=net, spatial=True, verbose=False).to(self.device).eval()

    def _t(self, img):
        x = self.torch.from_numpy(img).permute(2, 0, 1)[None].float().div(127.5).sub(1)
        return x.to(self.device)

    def map(self, a, b):
        with self.torch.inference_mode():
            return self.model(self._t(a), self._t(b))[0, 0].cpu().numpy()


def _region_mean(m, region):
    return float(m[region].mean()) if region is not None else float(m.mean())


def flow_pair(gt_t, gt_prev):
    """Flow t -> t-1 on GT, plus a forward-backward occlusion mask."""
    g1 = cv2.cvtColor(gt_t, cv2.COLOR_RGB2GRAY)
    g0 = cv2.cvtColor(gt_prev, cv2.COLOR_RGB2GRAY)
    args = (None, 0.5, 4, 21, 3, 5, 1.1, 0)
    fw = cv2.calcOpticalFlowFarneback(g1, g0, *args)  # g1(p) ~ g0(p + fw(p))
    bw = cv2.calcOpticalFlowFarneback(g0, g1, *args)
    h, w = g1.shape
    gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    mx, my = gx + fw[..., 0], gy + fw[..., 1]
    bw_w = cv2.remap(bw, mx, my, cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    diff = ((fw + bw_w) ** 2).sum(-1)
    mag = (fw ** 2).sum(-1) + (bw_w ** 2).sum(-1)
    inside = (mx >= 0) & (mx <= w - 1) & (my >= 0) & (my <= h - 1)
    valid = (diff < 0.01 * mag + 0.5) & inside
    return (mx, my), valid


def warp_error(cur, prev, maps, valid):
    warped = cv2.remap(prev.astype(np.float32), maps[0], maps[1], cv2.INTER_LINEAR,
                       borderMode=cv2.BORDER_REPLICATE)
    err = ((cur.astype(np.float32) - warped) / 255.0) ** 2
    err = err.mean(-1)
    return float(err[valid].mean()) if valid.any() else np.nan


def evaluate_tube(sr, gt, masks, lpips_model=None, flows=None, min_mask_frac=0.01):
    """sr, gt: lists of HxWx3 uint8. masks: list of HxW uint8 (0/255).

    Returns a dict of means over frames. Mask-level values skip frames whose
    mask covers < min_mask_frac of the crop (target briefly lost/occluded).
    """
    rows = {k: [] for k in ["psnr", "psnr_mask", "ssim", "ssim_mask",
                            "lpips", "lpips_mask", "tlpips", "warp", "warp_gt"]}
    for t, (s, g, m) in enumerate(zip(sr, gt, masks)):
        region = m > 127
        use_mask = region.mean() >= min_mask_frac
        sy, gy = rgb_to_y(s), rgb_to_y(g)
        rows["psnr"].append(psnr(sy, gy))
        rows["ssim"].append(ssim(sy, gy))
        if use_mask:
            rows["psnr_mask"].append(psnr(sy, gy, region))
            rows["ssim_mask"].append(ssim(sy, gy, region))
        if lpips_model is not None:
            lm = lpips_model.map(s, g)
            rows["lpips"].append(_region_mean(lm, None))
            if use_mask:
                rows["lpips_mask"].append(_region_mean(lm, region))
            if t > 0:
                d_sr = lpips_model.map(s, sr[t - 1]).mean()
                d_gt = lpips_model.map(g, gt[t - 1]).mean()
                rows["tlpips"].append(abs(float(d_sr) - float(d_gt)))
        if t > 0 and flows is not None:
            maps, valid = flows[t - 1]
            v = valid & region if use_mask else valid
            rows["warp"].append(warp_error(s, sr[t - 1], maps, v))
            rows["warp_gt"].append(warp_error(g, gt[t - 1], maps, v))
    out = {}
    for k, v in rows.items():
        v = [x for x in v if x is not None and not np.isnan(x)]
        out[k] = float(np.mean(v)) if v else np.nan
    for k in ("warp", "warp_gt"):  # report ×1e3 for readability
        out[k] = out[k] * 1e3 if not np.isnan(out[k]) else np.nan
    return out


In [ ]:
%%writefile tsr/vprior.py
"""Our method: restore a target tube with a FROZEN video-diffusion prior.

Training-free (option 1 on slide 10 of the prep deck). Nothing is trained; the
low-resolution target steers the frozen Stable Video Diffusion (SVD) model
through three mechanisms:

  1. Tube initialisation (SDEdit-style). The bicubic-upsampled LR tube is
     encoded by the frozen VAE and noised only part-way (`strength`). The
     denoiser then restores detail instead of inventing a new video, so the
     content and motion come from the input.
  2. Frame conditioning. SVD is an image-to-video model: every frame of a
     window is conditioned (CLIP embedding + concatenated VAE latent) on the
     window's first frame. Windows overlap by one frame, and window k>0 is
     conditioned on the ALREADY RESTORED last frame of window k-1, with that
     frame's latent used as its initialisation. Detail is carried forward
     instead of being re-invented every window, which limits flicker on clips
     longer than the model window.
  3. Low-frequency consistency. After decoding, the low frequencies are taken
     from the bicubic-upsampled LR and only the high frequencies from the prior
     (the colour/wavelet fix of StableSR [4]). The prior can add texture but
     cannot shift colour or structure away from what the LR shows.

All three temporal layers of SVD act across the window, which is the
motivation for a video prior instead of a per-frame image prior (sd_x4).
Interface matches tsr/baselines: process(list of LR uint8 frames) -> SR frames.
"""
import cv2
import numpy as np


def lowfreq_fix(sr, ref, sigma):
    """Replace the low frequencies of sr with those of ref (both HxWx3 uint8)."""
    if sigma <= 0:
        return sr
    sr_f, ref_f = sr.astype(np.float32), ref.astype(np.float32)
    blur = lambda x: cv2.GaussianBlur(x, (0, 0), sigma)  # noqa: E731
    return np.clip(blur(ref_f) + (sr_f - blur(sr_f)), 0, 255).astype(np.uint8)


def _work_size(h, w, target, mult=64):
    s = target / max(h, w)
    return (max(mult, int(round(w * s / mult)) * mult), max(mult, int(round(h * s / mult)) * mult))


class SVDTube:
    name = "ours_svd"

    def __init__(self, model_id="stabilityai/stable-video-diffusion-img2vid",
                 work_size=512, window=14, steps=25, strength=0.5, noise_aug=0.02,
                 motion_bucket=127, fps=7, fix_sigma=3.0, seed=0, decode_chunk=2,
                 offload=False, scale=4, **_):
        import torch
        from diffusers import StableVideoDiffusionPipeline
        self.torch = torch
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        kw = {"variant": "fp16"} if dtype == torch.float16 else {}
        self.pipe = StableVideoDiffusionPipeline.from_pretrained(model_id, torch_dtype=dtype, **kw)
        if offload and self.device == "cuda":
            self.pipe.enable_model_cpu_offload()
        else:
            self.pipe.to(self.device)
        if hasattr(self.pipe.unet, "enable_forward_chunking"):
            self.pipe.unet.enable_forward_chunking()  # lower peak memory, same output
        self.pipe.set_progress_bar_config(disable=True)
        for p in (self.pipe.unet, self.pipe.vae, self.pipe.image_encoder):
            p.requires_grad_(False)  # frozen prior
        self.cfg = dict(work_size=work_size, window=window, steps=steps, strength=strength,
                        noise_aug=noise_aug, motion_bucket=motion_bucket, fps=fps,
                        fix_sigma=fix_sigma, seed=seed, decode_chunk=decode_chunk)
        self.scale = scale

    # ------------------------------------------------------------------ core
    def _to_tensor(self, frames):
        x = self.torch.from_numpy(np.stack(frames)).permute(0, 3, 1, 2).float() / 127.5 - 1
        return x.to(self.device)

    def _run_window(self, init_frames, cond_frame):
        """init_frames: n uint8 frames at work size. Returns n uint8 frames."""
        from PIL import Image
        torch, pipe, c = self.torch, self.pipe, self.cfg
        dev, n = self.device, len(init_frames)
        H, W = init_frames[0].shape[:2]
        gen = torch.Generator(device="cpu").manual_seed(c["seed"])
        with torch.inference_mode():
            cond_pil = Image.fromarray(cond_frame)
            emb = pipe._encode_image(cond_pil, dev, 1, False)
            cond = pipe.video_processor.preprocess(cond_pil, height=H, width=W).to(dev)
            cond = cond + c["noise_aug"] * torch.randn(cond.shape, generator=gen).to(dev, cond.dtype)

            upcast = pipe.vae.dtype == torch.float16 and pipe.vae.config.force_upcast
            if upcast:
                pipe.vae.to(dtype=torch.float32)
            cond_lat = pipe._encode_vae_image(cond, dev, 1, False)
            x = self._to_tensor(init_frames).to(pipe.vae.dtype)
            z0 = torch.cat([pipe.vae.encode(x[i:i + 4]).latent_dist.mode()
                            for i in range(0, n, 4)]) * pipe.vae.config.scaling_factor
            if upcast:
                pipe.vae.to(dtype=torch.float16)

            cond_lat = cond_lat.to(emb.dtype).unsqueeze(1).repeat(1, n, 1, 1, 1)
            z0 = z0.to(emb.dtype).unsqueeze(0)  # (1, n, 4, h, w)
            added = pipe._get_add_time_ids(c["fps"] - 1, c["motion_bucket"], c["noise_aug"],
                                           emb.dtype, 1, 1, False).to(dev)

            sched = pipe.scheduler
            sched.set_timesteps(c["steps"], device=dev)
            start = min(max(int(round(c["steps"] * (1 - c["strength"]))), 0), c["steps"] - 1)
            ts = sched.timesteps[start:]
            if hasattr(sched, "set_begin_index"):
                sched.set_begin_index(start)
            noise = torch.randn(z0.shape, generator=gen).to(dev, z0.dtype)
            latents = sched.add_noise(z0, noise, ts[:1])

            for t in ts:
                inp = torch.cat([sched.scale_model_input(latents, t), cond_lat], dim=2)
                pred = pipe.unet(inp, t, encoder_hidden_states=emb, added_time_ids=added,
                                 return_dict=False)[0]
                latents = sched.step(pred, t, latents).prev_sample

            out = pipe.decode_latents(latents, n, c["decode_chunk"])  # (1, 3, n, H, W)
        out = (out[0].permute(1, 2, 3, 0).clamp(-1, 1) + 1) * 127.5
        return list(out.round().byte().cpu().numpy())

    def process(self, frames):
        c, s = self.cfg, self.scale
        h, w = frames[0].shape[:2]
        up = [cv2.resize(f, (w * s, h * s), interpolation=cv2.INTER_CUBIC) for f in frames]
        ws = _work_size(h * s, w * s, c["work_size"])
        work = [cv2.resize(f, ws, interpolation=cv2.INTER_CUBIC) for f in up]

        T, out, idx = len(frames), [None] * len(frames), 0
        while idx < T:
            first = 0 if idx == 0 else idx - 1  # overlap one restored frame
            ids = list(range(first, min(first + c["window"], T)))
            init = [out[j] if out[j] is not None else work[j] for j in ids]
            res = self._run_window(init, init[0])
            for j, r in zip(ids, res):
                if out[j] is None:
                    out[j] = r
            idx = ids[-1] + 1

        final = []
        for o, ref in zip(out, up):
            o = cv2.resize(o, (w * s, h * s), interpolation=cv2.INTER_AREA)
            final.append(lowfreq_fix(o, ref, c["fix_sigma"]))
        return final


In [ ]:
%%writefile tsr/targeted.py
"""End-to-end targeted SR on a whole clip (what the demo runs).

  LR clip + frame-0 box -> track -> square tube -> restore tube with a method
  -> paste back into the baseline-upscaled frame with a feathered border.

The background is the SAME baseline used for comparison, so baseline and ours
differ only inside the target tube: any difference you see is the targeting.
"""
import subprocess

import cv2
import numpy as np

from . import degrade, metrics, tubes


def make_lr(frames, preset="bicubic", seed=0, scale=4):
    """HD clip -> (LR clip, GT clip). GT is cropped to a multiple of `scale`."""
    h, w = frames[0].shape[:2]
    gt = [f[: h - h % scale, : w - w % scale] for f in frames]
    rng = np.random.default_rng(seed)
    params = degrade.sample_params(rng, preset)
    if params.get("crf") is not None and (gt[0].shape[0] // scale) % 2:
        params["crf"] = None  # H.264 needs even LR sizes
    return degrade.degrade_clip(gt, params, scale, rng), gt


def feather_weights(n, frac=0.15):
    """n x n weights: 1 inside, smooth ramp to 0 over `frac` of the side at the border."""
    ramp = max(1, int(n * frac))
    d = np.minimum.reduce(np.meshgrid(np.arange(n), np.arange(n))
                          + np.meshgrid(n - 1 - np.arange(n), n - 1 - np.arange(n)))
    t = np.clip(d / ramp, 0, 1)
    return (t * t * (3 - 2 * t)).astype(np.float32)[..., None]  # smoothstep


def plan_lr_tube(boxes_lr, lr_hw, margin=1.25, max_lr_side=128):
    plan = tubes.plan_window(boxes_lr, lr_hw, scale=1, margin=margin, align=8, min_side=16,
                             max_side=max_lr_side, pad=0)
    if plan is None:
        raise ValueError("Target too small: draw a box of at least ~16 LR pixels.")
    return plan


def restore_clip(lr, boxes_lr, method, baseline, scale=4, margin=1.25, max_lr_side=128,
                 feather=0.15):
    """Returns dict(ours, base, plan, tube_lr, tube_sr) with full-size frames."""
    plan = plan_lr_tube(boxes_lr, lr[0].shape[:2], margin, max_lr_side)
    tube_lr = [tubes.extract_padded(f, plan, i) for i, f in enumerate(lr)]
    tube_sr = method.process(tube_lr)
    base = baseline.process(lr)
    n = plan["side_raw"] * scale
    wgt = feather_weights(n, feather)
    ours = []
    for i, (b, t) in enumerate(zip(base, tube_sr)):
        if t.shape[0] != n:
            t = cv2.resize(t, (n, n), interpolation=cv2.INTER_CUBIC)
        x, y = plan["x0"][i] * scale, plan["y0"][i] * scale
        o = b.copy()
        region = o[y:y + n, x:x + n].astype(np.float32)
        o[y:y + n, x:x + n] = np.clip(t * wgt + region * (1 - wgt), 0, 255).astype(np.uint8)
        ours.append(o)
    return dict(ours=ours, base=base, plan=plan, tube_lr=tube_lr, tube_sr=tube_sr)


def tube_crops(frames, plan, scale=4):
    n = plan["side_raw"] * scale
    return [f[plan["y0"][i] * scale:plan["y0"][i] * scale + n,
              plan["x0"][i] * scale:plan["x0"][i] * scale + n] for i, f in enumerate(frames)]


def box_masks(boxes_lr, plan, scale=4):
    """Object-box masks inside each HR tube crop (uint8 0/255)."""
    n = plan["side_raw"] * scale
    out = []
    for i, (x0, y0, x1, y1) in enumerate(boxes_lr):
        m = np.zeros((n, n), np.uint8)
        ox, oy = plan["x0"][i], plan["y0"][i]
        a, b = int(max(0, (x0 - ox) * scale)), int(max(0, (y0 - oy) * scale))
        c, d = int(min(n, (x1 - ox) * scale)), int(min(n, (y1 - oy) * scale))
        m[b:d, a:c] = 255
        out.append(m)
    return out


def seam_score(frames, gt, plan, scale=4, band=3):
    """Mean |grad(out) - grad(gt)| in a thin band on the tube border (lower = less seam)."""
    n = plan["side_raw"] * scale
    vals = []
    for i, (f, g) in enumerate(zip(frames, gt)):
        x, y = plan["x0"][i] * scale, plan["y0"][i] * scale
        H, W = g.shape[:2]
        ring = np.zeros((H, W), bool)
        ring[max(0, y - band):y + n + band, max(0, x - band):x + n + band] = True
        ring[y + band:y + n - band, x + band:x + n - band] = False
        grads = []
        for img in (f, g):
            gr = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
            grads.append(np.hypot(cv2.Sobel(gr, cv2.CV_32F, 1, 0), cv2.Sobel(gr, cv2.CV_32F, 0, 1)))
        vals.append(float(np.abs(grads[0] - grads[1])[ring].mean()))
    return float(np.mean(vals))


def compare(result, gt, boxes_lr, lpips_model=None, scale=4):
    """Region metrics for baseline vs ours against GT (tube crops + object box)."""
    plan = result["plan"]
    gt_c = tube_crops(gt, plan, scale)
    masks = box_masks(boxes_lr, plan, scale)
    flows = [metrics.flow_pair(gt_c[t], gt_c[t - 1]) for t in range(1, len(gt_c))]
    rows = {}
    for name in ("base", "ours"):
        r = metrics.evaluate_tube(tube_crops(result[name], plan, scale), gt_c, masks,
                                  lpips_model, flows)
        r["seam"] = seam_score(result[name], gt, plan, scale)
        rows[name] = r
    return rows


def write_video(frames, path, fps=24):
    """H.264 mp4 that plays in browsers (cv2's mp4v often does not)."""
    h, w = frames[0].shape[:2]
    frames = [np.pad(f, ((0, h % 2), (0, w % 2), (0, 0)), mode="edge") for f in frames]
    h, w = frames[0].shape[:2]
    cmd = ["ffmpeg", "-loglevel", "error", "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
           "-s", f"{w}x{h}", "-r", str(fps), "-i", "-", "-c:v", "libx264", "-crf", "18",
           "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(path)]
    subprocess.run(cmd, input=b"".join(np.ascontiguousarray(f).tobytes() for f in frames),
                   check=True)
    return str(path)


def label(img, text):
    img = img.copy()
    cv2.rectangle(img, (0, 0), (min(img.shape[1], 12 + 11 * len(text)), 26), (0, 0, 0), -1)
    cv2.putText(img, text, (6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1,
                cv2.LINE_AA)
    return img


In [ ]:
%%writefile tsr/baselines/__init__.py
"""Baselines. Every method maps a list of LR frames (one tube) to SR frames ×4.

Methods are video-level (they get the whole tube) so temporal methods fit the
same interface. Heavy imports happen inside each class, so `bicubic` runs
without torch.
"""
from pathlib import Path

import cv2
import numpy as np

from ..io import to_uint8

CACHE = Path.home() / ".cache" / "tsr"


def _device():
    import torch
    return "cuda" if torch.cuda.is_available() else "cpu"


class Bicubic:
    name = "bicubic"

    def __init__(self, scale=4, **_):
        self.scale = scale

    def process(self, frames):
        h, w = frames[0].shape[:2]
        size = (w * self.scale, h * self.scale)
        return [cv2.resize(f, size, interpolation=cv2.INTER_CUBIC) for f in frames]


class RealESRGAN:
    """Real-ESRGAN x4plus, applied frame by frame (no temporal modelling).

    Loaded with `spandrel`, which avoids the broken basicsr/torchvision import.
    """
    name = "realesrgan"
    URL = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"

    def __init__(self, weights=None, batch=8, half=None, **_):
        import torch
        from spandrel import ModelLoader
        self.torch = torch
        self.device = _device()
        path = Path(weights) if weights else CACHE / "RealESRGAN_x4plus.pth"
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.hub.download_url_to_file(self.URL, str(path))
        self.model = ModelLoader().load_from_file(str(path)).to(self.device).eval()
        self.half = (self.device == "cuda") if half is None else half
        if self.half and self.model.supports_half:
            self.model.half()
        else:
            self.half = False
        self.batch = batch

    def process(self, frames):
        torch = self.torch
        out = []
        for i in range(0, len(frames), self.batch):
            x = torch.from_numpy(np.stack(frames[i:i + self.batch])).permute(0, 3, 1, 2)
            x = x.to(self.device).float().div(255)
            if self.half:
                x = x.half()
            with torch.inference_mode():
                y = self.model(x).float().clamp(0, 1)
            out += list(to_uint8(y.mul(255).permute(0, 2, 3, 1).cpu().numpy()))
        return out


class SDx4:
    """Stable Diffusion ×4 upscaler per frame: the Experiment 1 prior, kept as the
    image-prior ablation. Defaults match Exp 1's *targeted* branch (30 steps,
    noise_level 60, empty prompt). The same seed is used for every frame, which
    reduces (but does not remove) flicker.

    Exp 1 used guidance_scale 9 with an empty prompt. With CFG the conditional and
    unconditional branches then receive identical inputs, so guidance changes
    nothing and only doubles the cost. guidance=1.0 gives the same output faster."""
    name = "sd_x4"

    def __init__(self, steps=30, noise_level=60, guidance=1.0, prompt="", seed=0,
                 batch=2, offload=False,
                 model_id="stabilityai/stable-diffusion-x4-upscaler", **_):
        import torch
        from diffusers import StableDiffusionUpscalePipeline
        self.torch = torch
        self.device = _device()
        dtype = torch.float16 if self.device == "cuda" else torch.float32
        self.pipe = StableDiffusionUpscalePipeline.from_pretrained(model_id, torch_dtype=dtype)
        if offload and self.device == "cuda":
            self.pipe.enable_model_cpu_offload()  # as in Exp 1; slower, fits any T4
        else:
            self.pipe.to(self.device)
        self.pipe.enable_attention_slicing()
        self.pipe.set_progress_bar_config(disable=True)
        self.kw = dict(num_inference_steps=steps, noise_level=noise_level,
                       guidance_scale=guidance)  # guidance<=1 disables CFG (empty prompt anyway)
        self.prompt, self.seed, self.batch = prompt, seed, batch

    def process(self, frames):
        from PIL import Image
        torch = self.torch
        out = []
        for i in range(0, len(frames), self.batch):
            chunk = [Image.fromarray(f) for f in frames[i:i + self.batch]]
            gens = [torch.Generator(self.device).manual_seed(self.seed) for _ in chunk]
            res = self.pipe(prompt=[self.prompt] * len(chunk), image=chunk,
                            generator=gens, **self.kw).images
            out += [np.asarray(r) for r in res]
        return out


METHODS = {m.name: m for m in (Bicubic, RealESRGAN, SDx4)}


def build(name, **kw):
    if name == "ours_svd":
        from ..vprior import SVDTube
        return SVDTube(**kw)
    if name not in METHODS:
        raise KeyError(f"Unknown method {name}. Available: {sorted(METHODS) + ['ours_svd']} "
                       "(BasicVSR++/RealBasicVSR: scripts/run_mmagic_baseline.py)")
    return METHODS[name](**kw)


In [ ]:
%%writefile scripts/analyze_dataset.py
"""Analyse DAVIS 2017 annotations and pick the evaluation clips, before any GPU work.

Reads ONLY the annotation PNGs (fast, CPU, ~1 min for all of DAVIS) and answers:
  * how big are the targets, how fast do they move, how long are they visible?
  * which objects survive our tube rules (min/max side, margin, window length)?
  * which are genuinely *targeted* (tube much smaller than the frame) and not
    just "the whole frame"?
  * how long will the diffusion runs take for the chosen subset?

  python scripts/analyze_dataset.py --root data/raw/DAVIS --out analysis \
      --n-test 8 --n-val 4 --sec-per-frame ours_svd=4.0 sd_x4=1.5

Writes analysis/objects.csv, summary.md, distributions.png and the clip lists
analysis/selected_test.txt, selected_val.txt (lines "video:object") that
build_dataset.py --clip-list consumes. Selection uses only annotation
statistics, never results, so it cannot bias the comparison between methods.
"""
import argparse
import csv
import json
import sys
from pathlib import Path

import numpy as np
from PIL import Image

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from tsr import tubes  # noqa: E402


def object_stats(ann_dir, clip_len, margin, min_side, max_side):
    files = sorted(Path(ann_dir).glob("*.png"))
    masks = [np.array(Image.open(f)) for f in files]
    H, W = masks[0].shape[:2]
    ids = sorted({int(v) for m in masks for v in np.unique(m) if v not in (0, 255)})
    rows = []
    for oid in ids:
        boxes = [tubes.mask_to_box(m == oid) if (m == oid).sum() >= 64 else None for m in masks]
        vis = [b for b in boxes if b is not None]
        if not vis:
            continue
        b = np.array(vis, float)
        side = np.maximum(b[:, 2] - b[:, 0], b[:, 3] - b[:, 1])
        area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]) / (H * W)
        cx, cy = (b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2
        speed = np.hypot(np.diff(cx), np.diff(cy)) if len(b) > 1 else np.array([0.0])
        filled = tubes.fill_gaps(boxes, 3)
        runs = tubes.contiguous_runs(filled)
        longest = max((e - s for s, e in runs), default=0)
        status, tube_side, tube_frac, factor = "no_window", 0, np.nan, np.nan
        for s, e in runs:
            if e - s >= clip_len:
                plan = tubes.plan_window(filled[s:s + clip_len], (H, W), margin=margin,
                                         min_side=min_side, max_side=max_side)
                if plan is None:
                    status = "too_small"
                else:
                    status, tube_side = "ok", plan["side_raw"]
                    tube_frac, factor = plan["side_raw"] / min(H, W), plan["factor"]
                break
        rows.append(dict(video=Path(ann_dir).name, obj=oid, frames=len(masks), H=H, W=W,
                         visible=round(len(vis) / len(masks), 3), longest_run=longest,
                         box_side_med=round(float(np.median(side)), 1),
                         box_side_max=round(float(side.max()), 1),
                         area_frac_med=round(float(np.median(area)), 4),
                         speed_med=round(float(np.median(speed)), 2),
                         speed_p90=round(float(np.percentile(speed, 90)), 2),
                         status=status, tube_side=tube_side,
                         tube_frac=round(float(tube_frac), 3) if tube_side else "",
                         downscale=round(float(factor), 3) if tube_side else ""))
    return rows


def pick(rows, n, max_tube_frac, exclude_videos=()):
    """One object per video, eligible = ok + genuinely targeted; spread over motion."""
    cand = [r for r in rows if r["status"] == "ok" and r["tube_frac"] != ""
            and r["tube_frac"] <= max_tube_frac and r["video"] not in exclude_videos]
    best = {}
    for r in cand:  # per video keep the most "targeted" object (smallest tube)
        if r["video"] not in best or r["tube_frac"] < best[r["video"]]["tube_frac"]:
            best[r["video"]] = r
    pool = sorted(best.values(), key=lambda r: r["speed_med"])
    if len(pool) <= n:
        return pool
    idx = np.linspace(0, len(pool) - 1, n).round().astype(int)  # slow ... fast
    return [pool[i] for i in idx]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--root", required=True, help="DAVIS root (has Annotations/480p)")
    ap.add_argument("--res", default="480p")
    ap.add_argument("--out", default="analysis")
    ap.add_argument("--clip-len", type=int, default=27)
    ap.add_argument("--margin", type=float, default=1.25)
    ap.add_argument("--min-side", type=int, default=128)
    ap.add_argument("--max-side", type=int, default=512)
    ap.add_argument("--max-tube-frac", type=float, default=0.75,
                    help="tube side / frame short side; above this it is not 'targeted'")
    ap.add_argument("--n-test", type=int, default=8)
    ap.add_argument("--n-val", type=int, default=4)
    ap.add_argument("--degradations", type=int, default=2)
    ap.add_argument("--sec-per-frame", nargs="*", default=[],
                    help="method=seconds from _runtime.jsonl, for the time budget")
    args = ap.parse_args()

    root, out = Path(args.root), Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    sets = root / "ImageSets" / "2017"
    split_of = {v: "train" for v in (sets / "train.txt").read_text().split()}
    split_of.update({v: "val" for v in (sets / "val.txt").read_text().split()})

    rows = []
    for d in sorted((root / "Annotations" / args.res).iterdir()):
        for r in object_stats(d, args.clip_len, args.margin, args.min_side, args.max_side):
            r["split"] = split_of.get(r["video"], "?")
            rows.append(r)
    with open(out / "objects.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    # DAVIS val is the standard evaluation set; we never train, so train is unused.
    val_rows = [r for r in rows if r["split"] == "val"]
    test = pick(val_rows, args.n_test, args.max_tube_frac)
    val = pick(val_rows, args.n_val, args.max_tube_frac, {r["video"] for r in test})
    for name, sel in (("test", test), ("val", val)):
        (out / f"selected_{name}.txt").write_text(
            "".join(f"{r['video']}:{r['obj']}\n" for r in sel))

    def q(key, rs):
        v = np.array([r[key] for r in rs], float)
        return " / ".join(f"{x:.3g}" for x in np.percentile(v, [10, 50, 90])) if len(v) else "–"

    L = ["# DAVIS 2017 analysis\n",
         f"Tube rules: window {args.clip_len} frames, margin {args.margin}, "
         f"HR side {args.min_side}–{args.max_side} px, 'targeted' = tube ≤ "
         f"{args.max_tube_frac:.0%} of the frame's short side.\n",
         "| split | videos | objects | ok | too small | no full window | ok & targeted |",
         "|---|---|---|---|---|---|---|"]
    for sp in ("train", "val"):
        rs = [r for r in rows if r["split"] == sp]
        ok = [r for r in rs if r["status"] == "ok"]
        tg = [r for r in ok if r["tube_frac"] <= args.max_tube_frac]
        L.append(f"| {sp} | {len({r['video'] for r in rs})} | {len(rs)} | {len(ok)} | "
                 f"{sum(r['status'] == 'too_small' for r in rs)} | "
                 f"{sum(r['status'] == 'no_window' for r in rs)} | {len(tg)} |")
    L += ["\nPercentiles 10 / 50 / 90 (val objects):\n",
          f"* frames per video: {q('frames', val_rows)}",
          f"* target box long side (px @480p): {q('box_side_med', val_rows)}",
          f"* target area (fraction of frame): {q('area_frac_med', val_rows)}",
          f"* motion (px/frame, median per object): {q('speed_med', val_rows)}",
          f"* visible fraction: {q('visible', val_rows)}",
          "\n## Selected (one object per video, spread from slow to fast motion)\n",
          "| set | video:obj | box side | tube side | tube/frame | speed px/f |",
          "|---|---|---|---|---|---|"]
    for name, sel in (("test", test), ("val", val)):
        for r in sel:
            L.append(f"| {name} | {r['video']}:{r['obj']} | {r['box_side_med']} | "
                     f"{r['tube_side']} | {r['tube_frac']} | {r['speed_med']} |")

    n_tubes = (len(test) + 1) * args.degradations  # +1 = Exp 1 sign clip window
    frames = n_tubes * args.clip_len
    L += [f"\n## Time budget (test set: {len(test)} DAVIS + 1 Exp 1 window × "
          f"{args.degradations} degradations = {n_tubes} tubes, {frames} frames)\n"]
    for spec in args.sec_per_frame:
        m, s = spec.split("=")
        L.append(f"* {m}: {frames * float(s) / 3600:.2f} h on one GPU "
                 f"({frames * float(s) / 7200:.2f} h split over 2 GPUs)")
    if not args.sec_per_frame:
        L.append("* run `run_baseline.py --limit 1` per method, then pass "
                 "`--sec-per-frame method=sec` to fill this in.")
    (out / "summary.md").write_text("\n".join(L) + "\n")
    print("\n".join(L))

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
        for a, key, title in zip(ax, ("box_side_med", "speed_med", "tube_frac"),
                                 ("target box long side (px)", "motion (px/frame)",
                                  "tube side / frame short side")):
            vals = [r[key] for r in val_rows if r[key] != ""]
            a.hist(vals, bins=15, color="#c98a2b")
            sel_vals = [r[key] for r in test if r[key] != ""]
            a.plot(sel_vals, [0] * len(sel_vals), "v", color="#13202f", ms=8, label="selected")
            a.set_title(title, fontsize=10)
        ax[2].axvline(args.max_tube_frac, ls="--", color="gray")
        ax[0].legend(fontsize=8)
        fig.suptitle("DAVIS 2017 val objects", fontsize=11)
        fig.tight_layout()
        fig.savefig(out / "distributions.png", dpi=160)
    except ImportError:
        pass
    (out / "selection.json").write_text(json.dumps({"test": test, "val": val}, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/build_dataset.py
"""Build paired (LR tube, HR tube, mask) samples.

Examples
  # DAVIS 2017 with its ground-truth masks (no SAM 2 needed)
  python scripts/build_dataset.py --preset davis --root data/raw/DAVIS --out data/tubes

  # REDS with SAM 2 tracking objects found by YOLO on frame 0 (REDS4 = test)
  python scripts/build_dataset.py --preset reds --root data/raw/REDS --out data/tubes \
      --targets sam2_detect

  # CelebV-HQ (or any folder of clips) where the whole frame is the target
  python scripts/build_dataset.py --preset folder --name celebv --root data/raw/celebv \
      --targets whole --margin 1.0 --auto-split

Output: <out>/<split>/<sample_id>/{hr,lr,mask}/NNNNN.png + meta.json,
        and <out>/<split>.jsonl indexing the samples.
"""
import argparse
import json
import sys
import zlib
from pathlib import Path

import numpy as np
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from tsr import degrade, targets, tubes  # noqa: E402
from tsr.io import VID_EXT, read_frames, write_frames  # noqa: E402

REDS4 = {"000", "011", "015", "020"}  # standard REDS test clips (taken from the train set)


def hash_split(name, val=0.1, test=0.1):
    u = (zlib.crc32(name.encode()) % 10_000) / 10_000
    return "test" if u < test else "val" if u < test + val else "train"


def list_clips(args):
    root = Path(args.root)
    clips = []
    if args.preset == "davis":
        sets = root / "ImageSets" / "2017"
        train = (sets / "train.txt").read_text().split()
        val = sorted((sets / "val.txt").read_text().split())
        for v in train:
            clips.append(("train", v))
        for i, v in enumerate(val):  # DAVIS has no public test GT: halve val
            clips.append(("val" if i % 2 == 0 else "test", v))
        return [dict(dataset="davis", clip=v, split=s,
                     frames=root / "JPEGImages" / "480p" / v,
                     masks=root / "Annotations" / "480p" / v) for s, v in clips]

    if args.preset == "reds":
        def find(name):
            hits = [p for p in root.rglob(name) if p.is_dir()]
            return hits[0] if hits else None
        tr, va = find("train_sharp"), find("val_sharp")
        out = []
        if tr:
            for d in sorted(p for p in tr.iterdir() if p.is_dir()):
                out.append(dict(dataset="reds", clip=d.name, frames=d,
                                split="test" if d.name in REDS4 else "train"))
        if va:
            for d in sorted(p for p in va.iterdir() if p.is_dir()):
                out.append(dict(dataset="reds", clip=f"val{d.name}", frames=d, split="val"))
        if not out:
            raise SystemExit(f"No train_sharp/ or val_sharp/ under {root}")
        return out

    # generic folder: each subfolder of frames or video file is one clip
    items = sorted(p for p in root.iterdir() if p.is_dir() or p.suffix.lower() in VID_EXT)
    return [dict(dataset=args.name, clip=targets.safe_id(p.stem), frames=p,
                 split=hash_split(p.stem) if args.auto_split else args.split) for p in items]


def get_tracks(args, clip, frames, names, tracker):
    if args.targets == "masks":
        mdir = clip.get("masks") or Path(args.masks_root) / clip["clip"]
        return targets.masks_from_annotations(mdir, names)
    if args.targets == "whole":
        return targets.whole_frame_target(frames)
    if args.targets == "box_track":
        return targets.track_boxes_mil(frames, targets.boxes_from_json(args.boxes_json, clip["clip"]))
    if args.targets == "sam2_boxes":
        boxes0 = targets.boxes_from_json(args.boxes_json, clip["clip"])
    else:  # sam2_detect
        boxes0 = targets.detect_boxes(frames[0])
    return tracker.track(frames, boxes0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--preset", choices=["davis", "reds", "folder"], required=True)
    ap.add_argument("--root", required=True)
    ap.add_argument("--out", default="data/tubes")
    ap.add_argument("--name", default="custom", help="dataset name for --preset folder")
    ap.add_argument("--split", default="test", help="split for --preset folder")
    ap.add_argument("--auto-split", action="store_true", help="80/10/10 split by clip name")
    ap.add_argument("--only-splits", nargs="*", default=None, help="e.g. --only-splits val test")
    ap.add_argument("--clip-list", default=None,
                    help='lines "video" or "video:obj" (from analyze_dataset.py); others skipped')
    ap.add_argument("--list-split", default=None,
                    help="split name for --clip-list clips (e.g. test); default keeps preset split")
    ap.add_argument("--targets", default=None,
                    choices=["masks", "box_track", "sam2_boxes", "sam2_detect", "whole"])
    ap.add_argument("--masks-root", default=None)
    ap.add_argument("--boxes-json", default=None)
    ap.add_argument("--sam2-model", default="facebook/sam2.1-hiera-small")
    ap.add_argument("--degradation", nargs="+", default=["realworld", "bicubic"],
                    choices=list(degrade.PRESETS))
    ap.add_argument("--scale", type=int, default=4)
    ap.add_argument("--margin", type=float, default=1.25)
    ap.add_argument("--min-side", type=int, default=128, help="HR px; LR = /scale")
    ap.add_argument("--max-side", type=int, default=512)
    ap.add_argument("--clip-len", type=int, default=32, help="frames per tube window")
    ap.add_argument("--min-len", type=int, default=16)
    ap.add_argument("--max-windows", type=int, default=2, help="per target track")
    ap.add_argument("--max-frames", type=int, default=100, help="frames read per clip")
    ap.add_argument("--max-gap", type=int, default=3)
    ap.add_argument("--max-clips", type=int, default=None)
    ap.add_argument("--seed", type=int, default=643)
    args = ap.parse_args()

    if args.targets is None:
        args.targets = "masks" if args.preset == "davis" else "whole"
    tracker = None
    if args.targets.startswith("sam2"):
        tracker = targets.SAM2Tracker(args.sam2_model)

    clips = list_clips(args)
    allowed = None
    if args.clip_list:
        allowed = {}
        for line in Path(args.clip_list).read_text().split():
            v, _, o = line.partition(":")
            allowed.setdefault(v, set())
            if o:
                allowed[v].add(int(o))
        clips = [c for c in clips if c["clip"] in allowed]
        if args.list_split:
            for c in clips:
                c["split"] = args.list_split
    if args.only_splits:
        clips = [c for c in clips if c["split"] in args.only_splits]
    clips = clips[: args.max_clips]
    out_root = Path(args.out)
    out_root.mkdir(parents=True, exist_ok=True)
    stats = {"clips": 0, "samples": 0, "too_small": 0, "no_target": 0}

    for clip in tqdm(clips, desc="clips"):
        frames, names = read_frames(clip["frames"], max_frames=args.max_frames)
        if len(frames) < args.min_len:
            continue
        stats["clips"] += 1
        tracks = get_tracks(args, clip, frames, names, tracker)
        if allowed and allowed.get(clip["clip"]):
            tracks = {o: m for o, m in tracks.items() if o in allowed[clip["clip"]]}
        if not tracks:
            stats["no_target"] += 1
        for oid, masks in tracks.items():
            boxes = tubes.fill_gaps([tubes.mask_to_box(m) for m in masks], args.max_gap)
            for run in tubes.contiguous_runs(boxes):
                for w, (s, e) in enumerate(tubes.windows(*run, args.clip_len, args.min_len,
                                                         args.max_windows)):
                    plan = tubes.plan_window(boxes[s:e], frames[0].shape[:2], scale=args.scale,
                                             margin=args.margin, min_side=args.min_side,
                                             max_side=args.max_side)
                    if plan is None:
                        stats["too_small"] += 1
                        continue
                    hr_pad = [tubes.extract_padded(frames[s + i], plan, i) for i in range(e - s)]
                    hr = [tubes.unpad(f, plan["pad"]) for f in hr_pad]
                    mask = [tubes.extract_mask(masks[s + i] if masks[s + i] is not None
                                               else np.zeros(frames[0].shape[:2], bool), plan, i)
                            for i in range(e - s)]
                    for preset in args.degradation:
                        sid = f"{clip['dataset']}_{clip['clip']}_o{oid}_w{w}_{preset}"
                        rng = np.random.default_rng(args.seed + zlib.crc32(sid.encode()))
                        params = degrade.sample_params(rng, preset)
                        lr_pad = degrade.degrade_clip(hr_pad, params, args.scale, rng)
                        lr = [tubes.unpad(f, plan["pad"] // args.scale) for f in lr_pad]
                        d = out_root / clip["split"] / sid
                        fnames = names[s:e]
                        write_frames(d / "hr", hr, fnames)
                        write_frames(d / "lr", lr, fnames)
                        write_frames(d / "mask", [m * 255 for m in mask], fnames)
                        meta = dict(sample_id=sid, dataset=clip["dataset"], clip=clip["clip"],
                                    obj=oid, split=clip["split"], frames=fnames, start=s,
                                    degradation=params, plan=plan, scale=args.scale,
                                    hr_size=hr[0].shape[:2], lr_size=lr[0].shape[:2])
                        (d / "meta.json").write_text(json.dumps(meta, indent=1))
                        stats["samples"] += 1
    write_index(out_root)
    print(json.dumps(stats, indent=1))


def write_index(out_root):
    """Rebuild <split>.jsonl from what is on disk (safe to re-run the builder)."""
    for split_dir in sorted(p for p in out_root.iterdir() if p.is_dir()):
        rows = []
        for m in sorted(split_dir.glob("*/meta.json")):
            meta = json.loads(m.read_text())
            rows.append({"sample_id": meta["sample_id"], "path": str(m.parent),
                         "dataset": meta["dataset"],
                         "degradation": meta["degradation"]["preset"]})
        (out_root / f"{split_dir.name}.jsonl").write_text(
            "".join(json.dumps(r) + "\n" for r in rows))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/run_baseline.py
"""Run baselines on every tube of a split. Resumable: finished samples are skipped,
so a Colab disconnect costs at most one sample.

  python scripts/run_baseline.py --data data/tubes/test --methods bicubic realesrgan sd_x4
"""
import argparse
import json
import sys
import time
from pathlib import Path

from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from tsr import baselines  # noqa: E402
from tsr.io import image_files, read_dir, write_frames  # noqa: E402


def peak_mem_gb(reset=False):
    try:
        import torch
        if not torch.cuda.is_available():
            return None
        if reset:
            torch.cuda.reset_peak_memory_stats()
            return None
        return round(torch.cuda.max_memory_allocated() / 2**30, 2)
    except Exception:  # no torch / broken install: bicubic still runs
        return None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="split folder, e.g. data/tubes/test")
    ap.add_argument("--methods", nargs="+", required=True)
    ap.add_argument("--out", default="results")
    ap.add_argument("--filter", default="", help="only samples whose id contains this")
    ap.add_argument("--limit", type=int, default=None)
    ap.add_argument("--shard", default="0/1",
                    help="i/n: process every n-th sample from i (2 GPUs: 0/2 and 1/2)")
    ap.add_argument("--sd-steps", type=int, default=30)
    ap.add_argument("--sd-noise", type=int, default=60, help="Exp 1 targeted used 60")
    ap.add_argument("--sd-guidance", type=float, default=1.0)
    ap.add_argument("--sd-offload", action="store_true", help="CPU offload, as in Exp 1")
    ap.add_argument("--svd-strength", type=float, default=0.5,
                    help="fraction of the noise schedule re-run (0 = copy input, 1 = full generation)")
    ap.add_argument("--svd-steps", type=int, default=25)
    ap.add_argument("--svd-work-size", type=int, default=512)
    ap.add_argument("--svd-fix-sigma", type=float, default=3.0, help="0 disables the low-freq fix")
    ap.add_argument("--svd-offload", action="store_true")
    ap.add_argument("--tag", default="", help="suffix for the results folder, e.g. _s0.3")
    args = ap.parse_args()

    samples = sorted(p for p in Path(args.data).iterdir()
                     if (p / "meta.json").exists() and args.filter in p.name)[: args.limit]
    i, n = (int(v) for v in args.shard.split("/"))
    samples = samples[i::n]
    for name in args.methods:
        if name == "ours_svd":
            kw = dict(strength=args.svd_strength, steps=args.svd_steps,
                      work_size=args.svd_work_size, fix_sigma=args.svd_fix_sigma,
                      offload=args.svd_offload)
        else:
            kw = dict(steps=args.sd_steps, noise_level=args.sd_noise,
                      guidance=args.sd_guidance, offload=args.sd_offload)
        method = baselines.build(name, **kw)
        name = name + args.tag  # ablations get their own results folder
        log = Path(args.out) / name / f"_runtime{'' if n == 1 else f'_{i}'}.jsonl"
        log.parent.mkdir(parents=True, exist_ok=True)
        for s in tqdm(samples, desc=name):
            lr_files = image_files(s / "lr")
            out_dir = Path(args.out) / name / s.name
            if out_dir.exists() and len(image_files(out_dir)) == len(lr_files):
                continue
            lr = read_dir(s / "lr")
            peak_mem_gb(reset=True)
            t0 = time.time()
            sr = method.process(lr)
            dt = time.time() - t0
            exp = (lr[0].shape[0] * 4, lr[0].shape[1] * 4)
            assert sr[0].shape[:2] == exp, f"{name}: got {sr[0].shape[:2]}, expected {exp}"
            write_frames(out_dir, sr, [f.stem for f in lr_files])
            with open(log, "a") as fh:
                fh.write(json.dumps({"sample_id": s.name, "frames": len(lr),
                                     "sec": round(dt, 2), "sec_per_frame": round(dt / len(lr), 3),
                                     "peak_gb": peak_mem_gb()}) + "\n")
        del method


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/evaluate.py
"""Score every method folder in results/ against the HR tubes.

  python scripts/evaluate.py --data data/tubes/test --results results

Writes results/metrics_per_sample.csv and results/summary.md (one table per
dataset x degradation, mean over samples). GT flow is computed once per sample
and shared by all methods.
"""
import argparse
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from tsr import metrics  # noqa: E402
from tsr.io import read_dir  # noqa: E402

COLS = ["psnr", "ssim", "lpips", "psnr_mask", "ssim_mask", "lpips_mask", "tlpips", "warp"]
NR_COLS = ["clipiqa", "musiq"]  # no-reference, as in Experiment 1 (pyiqa)
HIGHER = {"psnr", "ssim", "psnr_mask", "ssim_mask", "clipiqa", "musiq"}


class NoRef:
    """CLIP-IQA and MUSIQ via pyiqa, the same metrics Experiment 1 reported."""

    def __init__(self):
        import pyiqa
        import torch
        self.torch = torch
        dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dev = dev
        self.m = {n: pyiqa.create_metric(n, device=dev) for n in NR_COLS}

    def __call__(self, frames):
        out = {n: [] for n in self.m}
        with self.torch.no_grad():
            for f in frames:
                x = self.torch.from_numpy(f).permute(2, 0, 1)[None].float().div(255).to(self.dev)
                for n, fn in self.m.items():
                    out[n].append(float(fn(x).item()))
        return {n: float(np.mean(v)) for n, v in out.items()}


def _nanmean(v):
    v = [x for x in v if not np.isnan(x)]
    return float(np.mean(v)) if v else np.nan


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--results", default="results")
    ap.add_argument("--methods", nargs="*", default=None, help="default: every folder")
    ap.add_argument("--no-lpips", action="store_true")
    ap.add_argument("--nr", action="store_true",
                    help="also CLIP-IQA and MUSIQ (pip install pyiqa); adds a GT row")
    args = ap.parse_args()

    res = Path(args.results)
    methods = args.methods or sorted(p.name for p in res.iterdir()
                                     if p.is_dir() and not p.name.startswith("_"))
    lp = None
    if not args.no_lpips:
        try:
            lp = metrics.LPIPS()
        except ImportError:
            print("lpips/torch not installed: skipping LPIPS and tLPIPS")

    nr = NoRef() if args.nr else None
    cols = COLS + (NR_COLS if nr else [])
    samples = sorted(p for p in Path(args.data).iterdir() if (p / "meta.json").exists())
    rows = []
    for s in tqdm(samples, desc="samples"):
        meta = json.loads((s / "meta.json").read_text())
        gt, masks = read_dir(s / "hr"), read_dir(s / "mask", grayscale=True)
        flows = [metrics.flow_pair(gt[t], gt[t - 1]) for t in range(1, len(gt))]
        base = {"sample_id": s.name, "dataset": meta["dataset"],
                "degradation": meta["degradation"]["preset"]}
        if nr:  # reference point: how the HR ground truth itself scores
            rows.append({"method": "_gt_hr", **base, **{c: np.nan for c in COLS + ["warp_gt"]},
                         **nr(gt)})
        for m in methods:
            d = res / m / s.name
            if not d.exists():
                continue
            sr = read_dir(d)
            if len(sr) != len(gt) or sr[0].shape != gt[0].shape:
                print(f"skip {m}/{s.name}: shape/length mismatch")
                continue
            r = metrics.evaluate_tube(sr, gt, masks, lp, flows)
            if nr:
                r.update(nr(sr))
            rows.append({"method": m, **base, **r})

    if not rows:
        raise SystemExit("No results found to evaluate.")
    with open(res / "metrics_per_sample.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)

    groups = defaultdict(lambda: defaultdict(list))
    for r in rows:
        groups[(r["dataset"], r["degradation"])][r["method"]].append(r)
    lines = ["# Region-only results (mean over tubes)\n",
             "Arrows: ↑ higher is better, ↓ lower is better. *_mask = inside the target mask. "
             "warp = warping error ×1e3 on GT flow; GT's own warp error is the floor.\n"]
    for (ds, deg), by_m in sorted(groups.items()):
        n = len(next(iter(by_m.values())))
        lines.append(f"\n## {ds} · {deg} degradation · {n} tubes\n")
        lines.append("| method | " + " | ".join(f"{c} {'↑' if c in HIGHER else '↓'}" for c in cols) + " |")
        lines.append("|" + "---|" * (len(cols) + 1))
        means = {m: {c: _nanmean([r[c] for r in rs]) for c in cols + ["warp_gt"]}
                 for m, rs in by_m.items()}
        for m in sorted(means):
            cells = ["–" if np.isnan(means[m][c]) else f"{means[m][c]:.4f}" if "ssim" in c
                     or "lpips" in c or c == "clipiqa" else f"{means[m][c]:.2f}" for c in cols]
            lines.append(f"| {m} | " + " | ".join(cells) + " |")
        floor = _nanmean([v["warp_gt"] for v in means.values()])
        lines.append(f"\nGT warp-error floor: {floor:.2f}\n")
    (res / "summary.md").write_text("\n".join(lines))
    print("\n".join(lines))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/make_figures.py
"""Qualitative figures for the report/slides from results/ folders.

  python scripts/make_figures.py --data data/tubes/test --results results \
      --methods bicubic realesrgan sd_x4 ours_svd --out figures

Per sample:
  <id>_frames.png    middle frame: LR | each method | GT
  <id>_xt.png        temporal profile: one pixel row stacked over time.
                     Straight, smooth streaks = temporally stable; jagged,
                     noisy streaks = flicker. The standard VSR flicker figure.
Plus figures/runtime.png: seconds per frame per method (from _runtime.jsonl).
"""
import argparse
import json
import sys
from pathlib import Path

import cv2
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from tsr.io import read_dir  # noqa: E402

PANEL = 256


def tag(img, text):
    img = cv2.resize(img, (PANEL, PANEL), interpolation=cv2.INTER_AREA if img.shape[0] > PANEL
                     else cv2.INTER_NEAREST)
    out = np.full((PANEL + 24, PANEL, 3), 255, np.uint8)
    out[24:] = img
    cv2.putText(out, text, (4, 17), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1, cv2.LINE_AA)
    return out


def xt_slice(frames, row):
    return np.stack([f[row] for f in frames])  # (T, W, 3)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--results", default="results")
    ap.add_argument("--methods", nargs="+", required=True)
    ap.add_argument("--out", default="figures")
    ap.add_argument("--filter", default="")
    ap.add_argument("--limit", type=int, default=6)
    args = ap.parse_args()
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    res = Path(args.results)

    samples = sorted(p for p in Path(args.data).iterdir()
                     if (p / "meta.json").exists() and args.filter in p.name)
    done = 0
    for s in samples:
        if not all((res / m / s.name).exists() for m in args.methods):
            continue
        gt = read_dir(s / "hr")
        lr = read_dir(s / "lr")
        mid = len(gt) // 2
        seqs = {m: read_dir(res / m / s.name) for m in args.methods}
        panels = [tag(lr[mid], "LR input")] + [tag(seqs[m][mid], m) for m in args.methods] \
            + [tag(gt[mid], "ground truth")]
        cv2.imwrite(str(out / f"{s.name}_frames.png"), cv2.cvtColor(np.hstack(panels),
                                                                   cv2.COLOR_RGB2BGR))
        mask = read_dir(s / "mask", grayscale=True)[mid] > 127
        row = int(np.nonzero(mask.any(1))[0].mean()) if mask.any() else gt[0].shape[0] // 2
        cols = [("ground truth", gt)] + [(m, seqs[m]) for m in args.methods]
        strips = []
        for name, fr in cols:
            sl = cv2.resize(xt_slice(fr, row), (PANEL, max(48, 4 * len(fr))),
                            interpolation=cv2.INTER_NEAREST)
            head = np.full((22, PANEL, 3), 255, np.uint8)
            cv2.putText(head, name, (4, 16), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
            strips.append(np.vstack([head, sl]))
        cv2.imwrite(str(out / f"{s.name}_xt.png"), cv2.cvtColor(np.hstack(strips),
                                                               cv2.COLOR_RGB2BGR))
        done += 1
        if done >= args.limit:
            break
    print(f"wrote figures for {done} samples to {out}")

    rt = {}
    for m in args.methods:
        rows = [json.loads(line) for f in (res / m).glob("_runtime*.jsonl")
                for line in f.read_text().splitlines() if line.strip()]
        if rows:
            rt[m] = (np.mean([r["sec_per_frame"] for r in rows]),
                     max((r["peak_gb"] or 0) for r in rows))
    if rt:
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt
            fig, ax = plt.subplots(figsize=(6, 3))
            ax.bar(list(rt), [v[0] for v in rt.values()], color="#c98a2b")
            for i, (sec, gb) in enumerate(rt.values()):
                ax.text(i, sec, f"{sec:.2f}s\n{gb:.1f} GB", ha="center", va="bottom", fontsize=8)
            ax.set_ylabel("seconds / frame")
            ax.set_title("Runtime on the target tube")
            fig.tight_layout()
            fig.savefig(out / "runtime.png", dpi=160)
        except ImportError:
            pass
        (out / "runtime.json").write_text(json.dumps(rt, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/boxes_exp1.json
{"6655841": [[600, 60, 180, 180]]}


In [ ]:
%%writefile tests/smoke_test.py
"""CPU-only end-to-end check: fake DAVIS -> tubes -> bicubic -> metrics.

  python tests/smoke_test.py
"""
import json
import shutil
import subprocess
import sys
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

ROOT = Path(__file__).resolve().parents[1]
TMP = ROOT / "_smoke"


def make_fake_davis(root, videos, T=24, H=240, W=320, side=110):
    rng = np.random.default_rng(0)
    for v in videos:
        bg = cv2.GaussianBlur(rng.integers(0, 255, (H, W, 3), dtype=np.uint8), (0, 0), 3)
        tex = rng.integers(0, 255, (side, side, 3), dtype=np.uint8)
        tex = cv2.resize(cv2.resize(tex, (side // 4, side // 4)), (side, side), cv2.INTER_NEAREST)
        for t in range(T):
            x, y = 20 + 6 * t, 40 + int(20 * np.sin(t / 4))
            img = bg.copy()
            img[y:y + side, x:x + side] = tex
            m = np.zeros((H, W), np.uint8)
            m[y:y + side, x:x + side] = 1
            (root / "JPEGImages/480p" / v).mkdir(parents=True, exist_ok=True)
            (root / "Annotations/480p" / v).mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(root / "JPEGImages/480p" / v / f"{t:05d}.jpg"), img)
            p = Image.fromarray(m, mode="P")
            p.putpalette([0, 0, 0, 255, 0, 0] + [0] * 762)
            p.save(root / "Annotations/480p" / v / f"{t:05d}.png")
    sets = root / "ImageSets/2017"
    sets.mkdir(parents=True, exist_ok=True)
    (sets / "train.txt").write_text(videos[0] + "\n")
    (sets / "val.txt").write_text("\n".join(videos[1:]) + "\n")


def run(*cmd):
    print("$", " ".join(cmd))
    subprocess.run([sys.executable, *cmd], cwd=ROOT, check=True)


if __name__ == "__main__":
    shutil.rmtree(TMP, ignore_errors=True)
    make_fake_davis(TMP / "DAVIS", ["aaa", "bbb", "ccc"])
    run("scripts/build_dataset.py", "--preset", "davis", "--root", str(TMP / "DAVIS"),
        "--out", str(TMP / "tubes"), "--clip-len", "16", "--min-len", "8")
    run("scripts/run_baseline.py", "--data", str(TMP / "tubes/test"), "--methods", "bicubic",
        "--out", str(TMP / "results"))
    run("scripts/evaluate.py", "--data", str(TMP / "tubes/test"), "--results",
        str(TMP / "results"), "--no-lpips")
    meta = json.loads(next((TMP / "tubes/test").glob("*/meta.json")).read_text())
    print("tube HR", meta["hr_size"], "LR", meta["lr_size"])
    assert meta["hr_size"][0] == 4 * meta["lr_size"][0]
    print("SMOKE TEST PASSED")


In [ ]:
%%writefile app.py
"""Targeted video SR demo (the live test interface for the viva).

  python app.py            # local
  python app.py --share    # public link from Colab

Flow (slide 19 of the prep deck): upload a clip -> click two corners of the
target on frame 0 (or type the box) -> Restore. Shows baseline vs ours side by
side in one synced video, a zoomed view of the target, and region metrics when
ground truth exists.

Two modes:
  "HD clip (has ground truth)" - the upload is treated as HR; we degrade it to
      LR ourselves, restore, and score against the original. Use for metrics.
  "Low-res clip (real world)"  - the upload IS the input; no metrics, only a
      visual comparison. Use for reviewer-supplied test clips.
"""
import argparse
import tempfile
import time

import cv2
import gradio as gr
import numpy as np

from tsr import baselines, targeted, targets
from tsr.io import read_frames

_CACHE = {}
OURS = ["ours_svd", "sd_x4", "realesrgan", "bicubic"]
BASES = ["bicubic", "realesrgan"]
MAX_LR_LONG_SIDE = 480  # real-world mode: bigger inputs are shrunk so ×4 output stays sane


def get_method(name, strength=0.5):
    key = (name, strength if name == "ours_svd" else None)
    if key not in _CACHE:
        for k in [k for k in _CACHE if k[0] == "ours_svd"] if name == "ours_svd" else []:
            del _CACHE[k]  # keep only one SVD in memory
        kw = {"strength": strength} if name == "ours_svd" else {}
        _CACHE[key] = baselines.build(name, **kw)
    return _CACHE[key]


def load(video, max_frames):
    frames, _ = read_frames(video, max_frames=int(max_frames))
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    cap.release()
    if not frames:
        raise gr.Error("Could not read any frames from this video.")
    return frames, fps


def on_upload(video, max_frames):
    if video is None:
        return None, None, [], 0, 0, 0, 0
    frames, _ = load(video, 1)
    h, w = frames[0].shape[:2]
    s = min(w, h) // 4
    return frames[0], frames[0], [], w // 2 - s // 2, h // 2 - s // 2, s, s


def draw_box(img, x, y, w, h):
    if img is None:
        return None
    out = img.copy()
    t = max(2, img.shape[1] // 300)
    cv2.rectangle(out, (int(x), int(y)), (int(x + w), int(y + h)), (255, 170, 0), t)
    return out


def on_click(frame0, pts, evt: gr.SelectData):
    pts = (pts or []) + [tuple(evt.index)]
    if len(pts) < 2:
        return gr.update(), pts, gr.update(), gr.update(), gr.update(), gr.update()
    (x0, y0), (x1, y1) = pts[-2:]
    x, y, w, h = min(x0, x1), min(y0, y1), abs(x1 - x0), abs(y1 - y0)
    return draw_box(frame0, x, y, w, h), [], x, y, w, h


def run(video, mode, x, y, w, h, max_frames, ours_name, base_name, tracker, degradation,
        strength, progress=gr.Progress()):
    if video is None:
        raise gr.Error("Upload a video first.")
    if w < 8 or h < 8:
        raise gr.Error("Draw a box first (click two corners on frame 0).")
    t0 = time.time()
    progress(0.05, desc="Reading frames")
    frames, fps = load(video, max_frames)
    hd = mode.startswith("HD")
    scale = 4
    if hd:
        progress(0.1, desc=f"Making LR input ({degradation})")
        lr, gt = targeted.make_lr(frames, degradation)
        box0 = [x / scale, y / scale, (x + w) / scale, (y + h) / scale]
    else:
        lr, gt = frames, None
        f = min(1.0, MAX_LR_LONG_SIDE / max(lr[0].shape[:2]))
        if f < 1:
            lr = [cv2.resize(fr, None, fx=f, fy=f, interpolation=cv2.INTER_AREA) for fr in lr]
        box0 = [x * f, y * f, (x + w) * f, (y + h) * f]

    progress(0.2, desc=f"Tracking target ({tracker})")
    if tracker == "SAM 2":
        if "sam2" not in _CACHE:
            _CACHE["sam2"] = targets.SAM2Tracker()
        masks = _CACHE["sam2"].track(lr, {1: box0})[1]
    else:
        masks = targets.track_boxes_mil(lr, {1: box0})[1]
    boxes = targets_to_boxes(masks, box0)

    progress(0.35, desc=f"Restoring the target with {ours_name}")
    res = targeted.restore_clip(lr, boxes, get_method(ours_name, strength), get_method(base_name),
                                scale=scale)
    progress(0.85, desc="Writing videos")

    plan = res["plan"]
    tmp = tempfile.mkdtemp()
    side = []
    for i, (b, o) in enumerate(zip(res["base"], res["ours"])):
        n = plan["side_raw"] * scale
        bx, by = plan["x0"][i] * scale, plan["y0"][i] * scale
        b = cv2.rectangle(b.copy(), (bx, by), (bx + n, by + n), (255, 170, 0), 2)
        o = cv2.rectangle(o.copy(), (bx, by), (bx + n, by + n), (255, 170, 0), 2)
        side.append(np.hstack([targeted.label(b, f"Baseline: {base_name} (full frame)"),
                               targeted.label(o, f"Ours: {ours_name} on target")]))
    sbs = targeted.write_video(_fit(side, 1920), f"{tmp}/side_by_side.mp4", fps)

    crops = {"LR input": [cv2.resize(t, (256, 256), interpolation=cv2.INTER_NEAREST)
                          for t in res["tube_lr"]],
             "Baseline": targeted.tube_crops(res["base"], plan, scale),
             "Ours": targeted.tube_crops(res["ours"], plan, scale)}
    if gt is not None:
        crops["Ground truth"] = targeted.tube_crops(gt, plan, scale)
    zoom = [np.hstack([targeted.label(cv2.resize(c[i], (256, 256), interpolation=cv2.INTER_AREA
                                                 if c[i].shape[0] > 256 else cv2.INTER_CUBIC), k)
                       for k, c in crops.items()]) for i in range(len(lr))]
    zv = targeted.write_video(zoom, f"{tmp}/zoom.mp4", fps)

    table = None
    if gt is not None:
        progress(0.92, desc="Computing region metrics")
        lp = _lpips()
        m = targeted.compare(res, gt, boxes, lp, scale)
        cols = [("PSNR (box) ↑", "psnr_mask"), ("SSIM (box) ↑", "ssim_mask"),
                ("LPIPS (box) ↓", "lpips_mask"), ("tLPIPS ↓", "tlpips"),
                ("Warp err ×1e3 ↓", "warp"), ("Seam ↓", "seam")]
        table = [[lbl, _fmt(m["base"][k]), _fmt(m["ours"][k])] for lbl, k in cols]
    msg = (f"{len(lr)} frames · LR {lr[0].shape[1]}×{lr[0].shape[0]} · target tube "
           f"{plan['side_raw']}px LR → {plan['side_raw'] * scale}px · {time.time() - t0:.0f}s")
    return sbs, zv, table, msg


def targets_to_boxes(masks, box0):
    from tsr.tubes import fill_gaps, mask_to_box
    boxes = fill_gaps([mask_to_box(m) for m in masks], max_gap=len(masks))
    last = box0
    out = []
    for b in boxes:  # if the tracker loses the target, hold the last box
        last = b if b is not None else last
        out.append(last)
    return out


def _fit(frames, max_w):
    w = frames[0].shape[1]
    if w <= max_w:
        return frames
    f = max_w / w
    return [cv2.resize(x, None, fx=f, fy=f, interpolation=cv2.INTER_AREA) for x in frames]


def _lpips():
    if "lpips" not in _CACHE:
        try:
            from tsr.metrics import LPIPS
            _CACHE["lpips"] = LPIPS()
        except ImportError:
            _CACHE["lpips"] = None
    return _CACHE["lpips"]


def _fmt(v):
    return "–" if v is None or np.isnan(v) else f"{v:.4f}" if abs(v) < 10 else f"{v:.2f}"


def build_ui():
    with gr.Blocks(title="Targeted Video SR · Team Pairadox") as demo:
        gr.Markdown("## Targeted super-resolution of content in video clips\n"
                    "IE 643 course project · Team Pairadox. Upload a clip, click two corners "
                    "of the target on frame 0, then press **Restore**.")
        frame0 = gr.State(None)
        pts = gr.State([])
        with gr.Row():
            with gr.Column(scale=1):
                video = gr.Video(label="Input clip")
                mode = gr.Radio(["HD clip (has ground truth)", "Low-res clip (real world)"],
                                value="HD clip (has ground truth)", label="Input type")
                with gr.Row():
                    x = gr.Number(label="x", precision=0)
                    y = gr.Number(label="y", precision=0)
                    w = gr.Number(label="w", precision=0)
                    h = gr.Number(label="h", precision=0)
                ours = gr.Dropdown(OURS, value="ours_svd", label="Method on the target")
                base = gr.Dropdown(BASES, value="bicubic", label="Baseline (whole frame)")
                with gr.Accordion("Settings", open=False):
                    frames_n = gr.Slider(8, 64, value=28, step=1, label="Frames (consecutive)")
                    tracker = gr.Radio(["MIL", "SAM 2"], value="MIL", label="Tracker")
                    degr = gr.Radio(["bicubic", "realworld"], value="bicubic",
                                    label="Degradation (HD mode)")
                    strength = gr.Slider(0.2, 0.9, value=0.5, step=0.05,
                                         label="Ours: strength (higher = more generated detail)")
                btn = gr.Button("Restore", variant="primary")
            with gr.Column(scale=2):
                img = gr.Image(label="Frame 0: click two corners of the target",
                               interactive=False)
                sbs = gr.Video(label="Baseline vs ours (synced)")
                zoom = gr.Video(label="Target close-up")
                table = gr.Dataframe(headers=["Metric (target region)", "Baseline", "Ours"],
                                     label="Metrics (HD mode only)")
                status = gr.Markdown()

        video.change(on_upload, [video, frames_n], [img, frame0, pts, x, y, w, h])
        img.select(on_click, [frame0, pts], [img, pts, x, y, w, h])
        for comp in (x, y, w, h):
            comp.submit(draw_box, [frame0, x, y, w, h], img)
        btn.click(run, [video, mode, x, y, w, h, frames_n, ours, base, tracker, degr, strength],
                  [sbs, zoom, table, status])
    return demo


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--share", action="store_true")
    args = ap.parse_args()
    build_ui().queue().launch(share=args.share)


## Part B · experiments


In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!pip -q install diffusers transformers accelerate spandrel lpips pyiqa gradio
!python tests/smoke_test.py | tail -1


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))


## 1 · Data: download, analyse, select


In [ ]:
os.makedirs('data/raw', exist_ok=True)
if not os.path.exists('data/raw/DAVIS'):
    !wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /kaggle/temp_davis.zip
    !unzip -q /kaggle/temp_davis.zip -d data/raw && rm /kaggle/temp_davis.zip
!python scripts/analyze_dataset.py --root data/raw/DAVIS --out analysis --clip-len {CLIP_LEN} --n-test {N_TEST} --n-val {N_VAL}


In [ ]:
from IPython.display import Image, Markdown, display
display(Image('analysis/distributions.png'))


In [ ]:
B = f"--clip-len {CLIP_LEN} --max-windows 1"
!python scripts/build_dataset.py --preset davis --root data/raw/DAVIS --out data/tubes --clip-list analysis/selected_test.txt --list-split test {B}
!python scripts/build_dataset.py --preset davis --root data/raw/DAVIS --out data/tubes --clip-list analysis/selected_val.txt --list-split val {B}
if EXP1_VIDEO:
    os.makedirs('data/raw/exp1', exist_ok=True); shutil.copy(EXP1_VIDEO, 'data/raw/exp1/')
    !python scripts/build_dataset.py --preset folder --name exp1 --root data/raw/exp1 --out data/tubes --targets box_track --boxes-json configs/boxes_exp1.json --max-frames 212 --max-windows 1 --clip-len {CLIP_LEN}
!wc -l data/tubes/*.jsonl


## 2 · Timing (one tube per method) → time budget


In [ ]:
import json, glob, numpy as np
for m in ['realesrgan', 'sd_x4', 'ours_svd']:
    !python scripts/run_baseline.py --data data/tubes/test --methods {m} --out results_timing --limit 1
spf = {}
for m in ['realesrgan', 'sd_x4', 'ours_svd']:
    rows = [json.loads(l) for f in glob.glob(f'results_timing/{m}/_runtime*.jsonl') for l in open(f)]
    spf[m] = float(np.mean([r['sec_per_frame'] for r in rows])); print(m, rows[-1])
spec = ' '.join(f'{k}={v:.3f}' for k, v in spf.items())
!python scripts/analyze_dataset.py --root data/raw/DAVIS --out analysis --clip-len {CLIP_LEN} --n-test {N_TEST} --n-val {N_VAL} --sec-per-frame {spec} | tail -5


## 3 · Choose the strength on VAL (never on test)
Pick the strength with the best LPIPS inside the target on the val tubes. Report this
protocol in the report; it is what makes the test numbers honest.


In [ ]:
import pandas as pd
for s in STRENGTHS:
    !python scripts/run_baseline.py --data data/tubes/val --methods ours_svd --svd-strength {s} --tag _s{s} --out results_val
!python scripts/evaluate.py --data data/tubes/val --results results_val
v = pd.read_csv('results_val/metrics_per_sample.csv').groupby('method')[['psnr_mask','lpips_mask','tlpips']].mean()
display(v)
BEST = float(v['lpips_mask'].idxmin().split('_s')[-1]); print('chosen strength:', BEST)


## 4 · Full test run on both GPUs


In [ ]:
import torch
N_GPU = max(1, torch.cuda.device_count())  # 2 on "T4 x2"; 1 on P100 or a pushed kernel
def two_gpus(args):
    procs = [subprocess.Popen(f"CUDA_VISIBLE_DEVICES={g} python scripts/run_baseline.py {args} --shard {g}/{N_GPU}",
                              shell=True) for g in range(N_GPU)]
    for p in procs: p.wait()
!python scripts/run_baseline.py --data data/tubes/test --methods bicubic --out results
two_gpus("--data data/tubes/test --methods realesrgan sd_x4 --out results")
two_gpus(f"--data data/tubes/test --methods ours_svd --svd-strength {BEST} --out results")
# ablation: no low-frequency fix (test set, same strength)
two_gpus(f"--data data/tubes/test --methods ours_svd --svd-strength {BEST} --svd-fix-sigma 0 --tag _nofix --out results")


## 5 · Evaluate, figures, package


In [ ]:
!python scripts/evaluate.py --data data/tubes/test --results results --nr
!python scripts/make_figures.py --data data/tubes/test --results results --methods bicubic realesrgan sd_x4 ours_svd --out figures --limit 10
display(Markdown(open('results/summary.md').read()))
for f in sorted(glob.glob('figures/*_frames.png'))[:3]: display(Image(f))


In [ ]:
!cd /kaggle/working && zip -qr outputs.zip tsr/results tsr/results_val tsr/figures tsr/analysis
!ls -lh /kaggle/working/outputs.zip
